# HACSA-ngspice tutorial

이 notebook은 JSON 설정을 불러와 HACSA를 실행하고 결과를 확인하는 가장 짧은 흐름을 보여준다. 설치와 deck 작성 규칙은 `README.md`를 먼저 참고한다.

## 1. 설정 불러오기

프로젝트 폴더에서 notebook을 열고 `sample_manual.json`을 불러온다. 설정 항목은 실행 식별자, deck, archive와 평가 횟수, FoM, design space 순서로 배치되어 있다. `reject_spec`, `target_spec`, `pre_weight`, `post_weight`의 각 index는 deck이 spec을 저장하는 순서와 같다.

In [ ]:
from json import load

with open('sample_manual.json', encoding='utf-8') as file:
    config = load(file)

config

## 2. 이번 실행의 값 변경

불러온 dictionary를 바꾸면 원본 JSON은 수정되지 않는다. 예제 결과와 겹치지 않도록 `run_name`을 바꾸고, 처음 확인할 때는 `max_evals`를 줄인다. 같은 `run_name`의 temp/result 폴더는 실행할 때 교체된다. 평가는 CMA-ES batch 단위이므로 실제 횟수는 `max_evals`를 조금 넘을 수 있다.

In [ ]:
config['run_name'] = 'tutorial'
config['max_evals'] = 200
config

## 3. 최적화 실행

`run(config)`은 `AutoCircuit`, `StoreParetoFront`, `CMAES`를 구성하고 최적화를 실행한 뒤 `circuit`과 `store`를 반환한다. FoM이 0 이상인 design을 찾으면 평가 횟수가 남아 있어도 종료한다.

In [ ]:
from hacsa import run

circuit, store = run(config)

## 4. 결과 확인

`spec_names`의 순서는 네 spec 배열의 순서를 확인하는 기준이다. 전체 결과는 `result_folder`의 `result_spec.csv`와 `param_0`, `param_1`, ...에 저장된다.

In [ ]:
{
    'spec_names': circuit.spec_names,
    'best_fom': store.best_fom,
    'saved_designs': store.size,
    'result_folder': store.result_folder,
}

## 5. Target과 weight 자동 설정

입력한 target과 완전한 weight pair는 그대로 사용하고, 빠진 쪽만 자동 설정한다. `sample_auto.json`은 둘 다 생략한 full-auto 예제이다. 기본적으로 solver 평가 전에 128개의 Sobol 표본을 사용하며 이 횟수는 `max_evals`에 포함되지 않는다. 표본 수를 바꿀 때만 `auto_fom`을 추가한다. 각 spec의 측정 실패값 `-1e10`은 해당 spec의 통계에서만 제외한다.

In [ ]:
with open('sample_auto.json', encoding='utf-8') as file:
    auto_config = load(file)

auto_config['run_name'] = 'tutorial_auto'
auto_config['max_evals'] = 200
auto_config

In [ ]:
auto_circuit, auto_store = run(auto_config)

## 다음 단계

자신의 회로를 연결할 때는 `README.md`의 Deck 작성 규칙과 Design Space 정의를 따른다. solver, FoM, store를 선택하거나 custom circuit이 필요하면 `ADVANCED_USE.md`의 계약으로 넘어간다.